# Cassava Disease Detection - GPU Training on Colab

This notebook trains the ConvNeXt model on GPU (V100 or A100) for faster training.

**Steps:**
1. Mount Google Drive
2. Clone repository and install dependencies
3. Upload training data OR use provided data
4. Run training (Phase 1 + Phase 2)
5. Download trained models

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully")

## Step 2: Clone Repository and Set Up

In [ ]:
import os
import shutil

# Clean up any previous attempts
if os.path.exists('/content/cassava'):
    shutil.rmtree('/content/cassava')

# Clone the repository
!git clone https://github.com/Ojerinde/Cassava-Disease-Detection-Using-CNN.git /content/cassava
os.chdir('/content/cassava')

# Download Git LFS files
!git lfs pull

print("\n✓ Repository cloned and LFS files downloaded")

## Step 3: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q timm torch torchvision fastai

print("✓ All dependencies installed")

## Step 4: Upload Training Data

**Option A:** If you have data in Google Drive, update the path below.
**Option B:** Upload data directly using the file manager (Colab will handle slow uploads gracefully).

Your data should have this structure:
```
data/
  ├── Cassava___bacterial_blight/
  ├── Cassava___brown_streak_disease/
  ├── Cassava___green_mottle/
  ├── Cassava___healthy/
  └── Cassava___mosaic_disease/
```

In [ ]:
import os

# Check if data folder exists locally
if os.path.exists('/content/cassava/data'):
    print("✓ Data folder found locally")
    # List data structure
    for folder in os.listdir('/content/cassava/data'):
        path = f'/content/cassava/data/{folder}'
        if os.path.isdir(path):
            count = len([f for f in os.listdir(path) if f.endswith(('.jpg', '.png', '.jpeg'))])
            print(f"  {folder}: {count} images")
else:
    print("⚠️  Data folder not found. Options:")
    print("   1. Upload data files via the folder icon on the left")
    print("   2. Copy from Google Drive if stored there")
    print("   3. Skip - training will fail but you can verify setup")

## Step 5: Verify GPU Availability

In [ ]:
import torch

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Make sure GPU runtime is enabled!")
    print("    Go to Runtime → Change Runtime Type → GPU")

## Step 6: Run Training

This will:
1. Load and validate your data
2. Create data loaders with stratified sampling
3. Run Phase 1: Train classification head (5 epochs - backbone frozen)
4. Run Phase 2: Fine-tune full model (5 epochs - backbone unfrozen)
5. Export final model with all weights

**On GPU:** ~20-30 minutes total
**On CPU:** ~2-3 hours

In [ ]:
os.chdir('/content/cassava')
!python model/train.py

## Step 7: Verify Training Completed

In [ ]:
import os

model_path = '/content/cassava/model/models'
files_generated = []

for file in ['model.pkl', 'weights.pth', 'classes.json', 'metrics.json']:
    full_path = os.path.join(model_path, file)
    if os.path.exists(full_path):
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        files_generated.append((file, size_mb))
        print(f"✓ {file} ({size_mb:.1f} MB)")
    else:
        print(f"✗ {file} - NOT FOUND")

if len(files_generated) == 4:
    print("\n✓✓✓ Training completed successfully! All model files ready for download.")
else:
    print("\n⚠️  Some files missing. Check training output above for errors.")

## Step 8: Download Trained Models

The files below can be downloaded automatically or copied to Google Drive.

In [ ]:
from google.colab import files

model_dir = '/content/cassava/model/models'

print("Downloading model files...\n")

files_to_download = [
    'model.pkl',
    'weights.pth',
    'classes.json',
    'metrics.json'
]

for file in files_to_download:
    file_path = f'{model_dir}/{file}'
    if os.path.exists(file_path):
        print(f"Downloading {file}...")
        files.download(file_path)
    else:
        print(f"Skipping {file} (not found)")

print("\n✓ Download complete!")

## Alternative: Copy to Google Drive

If downloads are slow, save to Google Drive instead:

In [ ]:
import shutil
import os

# Create folder in Drive
drive_models = '/content/drive/MyDrive/cassava_models'
os.makedirs(drive_models, exist_ok=True)

model_dir = '/content/cassava/model/models'

print(f"Copying models to: {drive_models}\n")

for file in ['model.pkl', 'weights.pth', 'classes.json', 'metrics.json']:
    src = f'{model_dir}/{file}'
    dst = f'{drive_models}/{file}'
    if os.path.exists(src):
        print(f"Copying {file}...")
        shutil.copy2(src, dst)
        print(f"  ✓ Saved to Google Drive")
    else:
        print(f"✗ {file} not found")

print(f"\n✓ Models saved to Google Drive: {drive_models}")

## Next Steps

1. **Download** the model files from above (or access from Google Drive)
2. **Replace** in your local project:
   ```
   model/models/model.pkl     ← your downloaded file
   model/models/weights.pth   ← your downloaded file
   model/models/classes.json  ← your downloaded file
   model/models/metrics.json  ← your downloaded file
   ```
3. **Restart** the backend: `python -m uvicorn backend.main:app`
4. **Test** with the frontend - predictions should now have high confidence!